# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [1]:
from google.colab import userdata
HF_TOKEN=userdata.get('hf_key')

In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- What one row means for your lane: One row represents one page-query pair for one reporting period (month). Each row summarizes how a specific page performed for a specific search query during that month.

- Which table(s) you'll use: I'll be using the ***fact_content_daily_performance(78.8M)*** and the ***dim_content(520k)*** and ***fact_query_90d*** Tables

- Which time window: I'll pick September, October, and November 2025

- What you'd predict or rank (label or proxy): The model predicts the page's expected click-through rate (CTR). Needed for calculating the opportunity i.e. (Expected_CTR - Actual_CTR)

- One thing you deliberately exclude: I deliberately exclude the click column. the click column with the impression column is used for calculating the cTR. including the click column will leak the CTR and affect the accuracy of our model.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- Features: ==>>> [main intent, impression, average_position, search_volume, content_type]

- label: ==>>> [Click_Through_Rate (CTR)] --> clicks/impression

- exclude: ===>>> [Clicks]

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The grain (one row really is what you said),
your slice’s row count and date span,
availability — filter with IS TRUE and show how many rows survive

- The Grain ==>> Checking if a row really contain information about a single content over the last 30days


In [5]:
data = con.sql(f"""SELECT
    COUNT(*) AS rows,
    COUNT(content_hash_id) AS unique_content
FROM {TABLES['fact_daily']}
WHERE report_date > '2025-08-31'::DATE - INTERVAL 60 DAY

""").df()

print(f'{len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1 rows


,rows,unique_content
0,77603955,77603955


- Slice's row count -->>

In [8]:
data = con.sql(f"""SELECT
    COUNT(*) AS row_count,
    '2025-08-31'::DATE AS youngest_content,
    '2025-08-31'::DATE - INTERVAL 2 MONTH AS oldest_content
FROM {TABLES['fact_daily']}
WHERE report_date > '2025-08-31'::DATE - INTERVAL 60 DAY
""").df()

print(f'{len(data):,} rows')
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1 rows


,row_count,youngest_content,oldest_content
0,77603955,2025-08-31,2025-06-30


- Checking for availability -->

In [9]:
data_aval = con.sql(f"""SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE ga4_data_available IS TRUE
""").df()

print(f'{len(data_aval):,} rows')
data_aval.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1 rows


,available_rows
0,2816455


### Picking the features

In [ ]:
## features ==> client_hash_id, content_hash_id, main_intent,
#impressions_30d, avg_position_30d, search_volume_30,
#content_type, clicks_30d

data_features = con.sql(f"""
  WITH bounds AS (
        SELECT '2025-08-31'::DATE AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
    SELECT
        q.client_hash_id,
        q.content_hash_id,
        d.main_intent AS intent,
        d.content_type AS content_type,
        SUM(CASE WHEN q.report_date >  b.end_d - INTERVAL 30 DAY THEN q.gsc_impressions ELSE 0 END) AS impression_last30d,
        SUM(CASE WHEN q.report_date <= b.end_d - INTERVAL 30 DAY THEN q.gsc_impressions ELSE 0 END) AS impression_prev30d,
        SUM(CASE WHEN q.report_date >  b.end_d - INTERVAL 30 DAY THEN q.gsc_clicks ELSE 0 END)      AS clicks_30d,
        AVG(CASE WHEN q.report_date >  b.end_d - INTERVAL 30 DAY THEN q.gsc_avg_position ELSE 0 END)       AS avg_pos_30d,
        MAX(d.search_volume) AS search_volume, -- Changed to MAX as search_volume is likely a static content attribute
        COALESCE(
            CAST(SUM(CASE WHEN q.report_date >  b.end_d - INTERVAL 30 DAY THEN q.gsc_clicks ELSE 0 END) AS DOUBLE) /
            NULLIF(SUM(CASE WHEN q.report_date >  b.end_d - INTERVAL 30 DAY THEN q.gsc_impressions ELSE 0 END), 0),
            0
        ) * 100 AS ctr_label -- Corrected COALESCE syntax and multiplication
    FROM {TABLES['fact_daily']} q, bounds b
    JOIN {TABLES['dim_content']} d
        ON q.content_hash_id = d.content_hash_id
    WHERE q.report_date > b.end_d - INTERVAL 60 DAY
    GROUP BY
        q.client_hash_id,
        q.content_hash_id,
        d.main_intent,
        d.content_type
    HAVING impression_prev30d >= 70
    )
    SELECT * FROM windowed
""").df()

print(f'{len(data_features):,} content items with enough history')
data_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
data_features = con.sql(f"""
WITH bounds AS (
    SELECT DATE '2025-08-31' AS end_d
),

windowed AS (
    SELECT
        q.client_hash_id,
        q.content_hash_id,

        ANY_VALUE(d.main_intent) AS main_intent,
        ANY_VALUE(d.content_type) AS content_type,
        ANY_VALUE(d.search_volume) AS search_volume,

        -- Previous 30 days (days -60 to -31)
        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY AND q.report_date <= b.end_d - INTERVAL 30 DAY
                THEN q.gsc_impressions ELSE 0 END) AS impressions_prev30d,

        -- Last 30 days (days -30 to end_d)
        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_impressions ELSE 0 END) AS impressions_last30d,

        SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_clicks
                ELSE 0 END) AS clicks_30d,

        AVG(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY THEN q.gsc_avg_position END
        ) AS avg_position_30d,

        100.0 *
        COALESCE(CAST(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                        THEN q.gsc_clicks ELSE 0 END) AS DOUBLE) /
            NULLIF(SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 30 DAY
                        THEN q.gsc_impressions ELSE 0 END), 0), 0) AS ctr_label

    FROM {TABLES['fact_daily']} q
    CROSS JOIN bounds b
    JOIN {TABLES['dim_content']} d
      ON q.content_hash_id = d.content_hash_id

    WHERE q.report_date > b.end_d - INTERVAL 60 DAY

    GROUP BY q.client_hash_id, q.content_hash_id

    HAVING SUM(CASE WHEN q.report_date > b.end_d - INTERVAL 60 DAY
                 AND q.report_date <= b.end_d - INTERVAL 30 DAY
                THEN q.gsc_impressions ELSE 0 END) >= 70
)

SELECT *
FROM windowed
""").df()

print(f"{len(data_features):,} content items with enough history")
data_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

15,310 content items with enough history


,client_hash_id,content_hash_id,main_intent,content_type,search_volume,impressions_prev30d,impressions_last30d,clicks_30d,avg_position_30d,ctr_label
0,client_fef1a8f436438636,content_ca2bdc59bd4b72c6,transactional,keyword article,10,405.0,16101.0,32.0,8.189929,0.198745
1,client_fef1a8f436438636,content_caafa0d51267e92b,informational,keyword article,10,304.0,12982.0,8.0,34.630552,0.061624
2,client_fef1a8f436438636,content_cb547c396b4bf957,informational,keyword article,27100,397.0,40094.0,26.0,7.484258,0.064848
3,client_fef1a8f436438636,content_cbefcde68fbf750c,commercial,keyword article,30,151.0,1367.0,2.0,26.177942,0.146306
4,client_fef1a8f436438636,content_cc083944bfc73d07,transactional,keyword article,20,308.0,1813.0,2.0,16.254962,0.110314


In [9]:
data_features.head(20)

,client_hash_id,content_hash_id,main_intent,content_type,search_volume,impressions_prev30d,impressions_last30d,clicks_30d,avg_position_30d,ctr_label
0,client_fef1a8f436438636,content_ca2bdc59bd4b72c6,transactional,keyword article,10,405.0,16101.0,32.0,8.189929,0.198745
1,client_fef1a8f436438636,content_caafa0d51267e92b,informational,keyword article,10,304.0,12982.0,8.0,34.630552,0.061624
2,client_fef1a8f436438636,content_cb547c396b4bf957,informational,keyword article,27100,397.0,40094.0,26.0,7.484258,0.064848
3,client_fef1a8f436438636,content_cbefcde68fbf750c,commercial,keyword article,30,151.0,1367.0,2.0,26.177942,0.146306
4,client_fef1a8f436438636,content_cc083944bfc73d07,transactional,keyword article,20,308.0,1813.0,2.0,16.254962,0.110314
5,client_fef1a8f436438636,content_cc9732f0da1d8d2d,informational,keyword article,49500,579.0,32317.0,1.0,20.258793,0.003094
6,client_fef1a8f436438636,content_cd12aa5acbd3ba94,informational,keyword article,10,228.0,2074.0,2.0,22.358373,0.096432
7,client_fef1a8f436438636,content_d120eb606fe94e31,commercial,keyword article,20,214.0,3488.0,12.0,12.300431,0.344037
8,client_fef1a8f436438636,content_d358f57fbb8d0abd,transactional,keyword article,320,165.0,865.0,1.0,23.225432,0.115607
9,client_fef1a8f436438636,content_d80f1acaf1880683,informational,keyword article,20,301.0,4724.0,5.0,11.323335,0.105843


### Why the Features?

- **Avg_Position** --> Knowable because average search position has already been observed during the historical window.

- **Impressions_30d** --> Knowable because impressions are historical measurements, not future outcomes.

- **Search_Volume** --> Knowable because keyword demand is external metadata collected independently of page performance.

- **Main_Intent** --> Knowable because search intent is inferred from the query before optimization decisions are made.

- **Content_Type** --> Knowable because the content type is determined when the page is created.

### Training a model including leakage

 - I'll include clicks -- This data is used to derive out label

In [10]:
data_features.content_type.value_counts()

,count
content_type,
keyword article,15139
feedly article,171


In [7]:
data_features.main_intent.value_counts()

,count
main_intent,
informational,6906
transactional,5504
commercial,2716
navigational,7


In [12]:
##Training a model model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

## X, y
X, y = data_features.drop(columns=["client_hash_id","content_hash_id",'ctr_label']), data_features['ctr_label']

##splitting dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## converting categorical data to int
X_train = pd.get_dummies(X_train,)
X_test = pd.get_dummies(X_test)

##model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

## evaluation
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"Root Mean Squared Error: {np.sqrt(mse)}")

Mean Squared Error: 0.0020593806918356035
Root Mean Squared Error: 0.045380399864210136


### Removing the leaked dataset

In [13]:
##Training a model model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

## X, y
X, y = data_features.drop(columns=["client_hash_id","content_hash_id",'clicks_30d','ctr_label']), data_features['ctr_label']

##splitting dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## converting categorical data to int
X_train = pd.get_dummies(X_train,)
X_test = pd.get_dummies(X_test)

##model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

## evaluation
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"Root Mean Squared Error: {np.sqrt(mse)}")

Mean Squared Error: 0.09288243551498401
Root Mean Squared Error: 0.30476619811748157


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset can identify patterns and associations, but it cannot explain why those patterns occur.

Specifically, this data cannot tell us:

- Whether changing a page title, meta description, or content will cause CTR to increase.
- How Google's ranking or search algorithms make decisions.
- The impact of SERP features (featured snippets, AI Overviews, knowledge panels, ads) unless those features are explicitly captured in the data.
- Whether changes in CTR are due to competitor activity, seasonality, news events, or shifts in user intent.
- User-level behavior, such as why an individual chose to click or not click a search result.
- Long-term trends or seasonal effects, since this analysis is based on a limited historical window.

As a result, the model should be used as a decision-support tool. It can rank pages that appear to underperform relative to similar pages, helping teams prioritize manual review. It cannot prove that optimizing a recommended page will increase clicks or predict how Google will rank pages in the future.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.